In [1]:
from torchvision import transforms
from transformers import FlavaModel,AutoProcessor
from torch.utils.data import DataLoader
import torch
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import numpy as np
from modules import FLAVAExtractor,HeadClassifierFLAVAModel,Train,creation_dataframe,CreationFLAVADataset,CreationProcessedDataset,FLAVACollateFunction

import sys, os
sys.path.append(os.path.abspath(".."))

from CLIP_model.modules import CreationProcessedDataset as CreationClipDataset


In [2]:
#creation of the dataframes
train_df=creation_dataframe("../data/train.jsonl")
val_df=creation_dataframe("../data/dev.jsonl")

In [3]:
#creation of the datasets used for the FLAVA forward
train_FLAVA_dataset=CreationFLAVADataset(train_df)
val_FLAVA_dataset=CreationFLAVADataset(val_df)

In [4]:
#processor used to process the input data before the FLAVA forward
processor=AutoProcessor.from_pretrained("facebook/flava-full")

In [5]:
flava_model=FlavaModel.from_pretrained("facebook/flava-full")
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
batch_size=32

In [6]:
collate_function_object=FLAVACollateFunction(processor)

In [7]:
#creation of the dataloaders for the FLAVA forward and process of the input data
train_FLAVA_dataloader=DataLoader(train_FLAVA_dataset,collate_fn=collate_function_object.collate_fn,batch_size=batch_size,shuffle=False)
val_FLAVA_dataloader=DataLoader(val_FLAVA_dataset,collate_fn=collate_function_object.collate_fn,batch_size=batch_size,shuffle=False)

In [8]:
flava_extractor=FLAVAExtractor(flava_model=flava_model,device=device)

In [9]:
#extraction of the FLAVA embeddings
#train_flava_data=flava_extractor.get_embeddings(train_FLAVA_dataloader,"./modules/flava_embeddings","train")
#val_flava_data=flava_extractor.get_embeddings(train_FLAVA_dataloader,"./modules/flava_embeddings","val")

In [9]:
train_flava_embeddings=torch.load("./modules/flava_embeddings/train_flava_embeddings.pt")
val_flava_embeddings=torch.load("./modules/flava_embeddings/val_flava_embeddings.pt")

In [10]:
train_clip_embeddings=torch.load("../CLIP_model/modules/clip_embeddings/train_clip_embeddings.pt")
val_clip_embeddings=torch.load("../CLIP_model/modules/clip_embeddings/val_clip_embeddings.pt")

In [11]:
train_all_embeddings=train_flava_embeddings.copy()
train_all_embeddings.update(train_clip_embeddings)
val_all_embeddings=val_flava_embeddings.copy()
val_all_embeddings.update(val_clip_embeddings)

In [12]:
train_dataset=CreationProcessedDataset(train_all_embeddings)
val_dataset=CreationProcessedDataset(val_all_embeddings)
train_dataloader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=batch_size,shuffle=True,drop_last=True)

In [13]:
print(train_all_embeddings["multimodal_embeddings"].shape)

torch.Size([8500, 326, 768])


In [14]:
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight, dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [15]:
#Training without CLIP embeddings and using FLAVA pooler embeddings
model=HeadClassifierFLAVAModel()
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=1e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/pooler_only_savings")

2026-03-12 17:23:59.071 | INFO     | modules.train:run_training:114 - Epoch 0 :
2026-03-12 17:24:20.695 | INFO     | modules.train:run_training:211 - Epoch 0: Train Loss = 0.7738371025841191
2026-03-12 17:24:20.755 | INFO     | modules.train:run_training:212 - Epoch 0: Train Accuracy = 0.6367924528301887
2026-03-12 17:24:20.757 | INFO     | modules.train:run_training:213 - Epoch 0: Train F1 = 0.5110316039460611
2026-03-12 17:24:20.760 | INFO     | modules.train:run_training:215 - Epoch 0: Validation Loss = 0.8648374597231547
2026-03-12 17:24:20.761 | INFO     | modules.train:run_training:216 - Epoch 0: Validation Accuracy = 0.5
2026-03-12 17:24:20.762 | INFO     | modules.train:run_training:217 - Epoch 0: Validation F1 = 0.3333333333333333
2026-03-12 17:24:21.007 | INFO     | modules.train:run_training:114 - Epoch 1 :
2026-03-12 17:24:37.078 | INFO     | modules.train:run_training:211 - Epoch 1: Train Loss = 0.7107569233426508
2026-03-12 17:24:37.144 | INFO     | modules.train:run_trai

In [17]:
#Training with CLIP embeddings and using FLAVA pooler embeddings
model=HeadClassifierFLAVAModel(with_clip_image=True,with_clip_text=True)
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=1e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/pooler_with_clip_savings",with_clip=True)

2026-03-12 17:27:08.812 | INFO     | modules.train:run_training:114 - Epoch 0 :
2026-03-12 17:27:12.240 | INFO     | modules.train:run_training:211 - Epoch 0: Train Loss = 0.7242236101402426
2026-03-12 17:27:12.240 | INFO     | modules.train:run_training:212 - Epoch 0: Train Accuracy = 0.5679245283018868
2026-03-12 17:27:12.240 | INFO     | modules.train:run_training:213 - Epoch 0: Train F1 = 0.547953923949787
2026-03-12 17:27:12.240 | INFO     | modules.train:run_training:215 - Epoch 0: Validation Loss = 0.7560108502705892
2026-03-12 17:27:12.256 | INFO     | modules.train:run_training:216 - Epoch 0: Validation Accuracy = 0.49166666666666664
2026-03-12 17:27:12.256 | INFO     | modules.train:run_training:217 - Epoch 0: Validation F1 = 0.4176503200276207
2026-03-12 17:27:12.351 | INFO     | modules.train:run_training:114 - Epoch 1 :
2026-03-12 17:27:15.677 | INFO     | modules.train:run_training:211 - Epoch 1: Train Loss = 0.6407365142174487
2026-03-12 17:27:15.678 | INFO     | modules

In [20]:
#Training without CLIP embeddings and using FLAVA multimodal embeddings
model=HeadClassifierFLAVAModel()
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=1e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/multimodal_only_savings",multimodal=True)

2026-03-12 17:33:55.332 | INFO     | modules.train:run_training:114 - Epoch 0 :
2026-03-12 17:33:58.159 | INFO     | modules.train:run_training:211 - Epoch 0: Train Loss = 0.769842326866006
2026-03-12 17:33:58.160 | INFO     | modules.train:run_training:212 - Epoch 0: Train Accuracy = 0.38254716981132075
2026-03-12 17:33:58.160 | INFO     | modules.train:run_training:213 - Epoch 0: Train F1 = 0.28489077155826287
2026-03-12 17:33:58.160 | INFO     | modules.train:run_training:215 - Epoch 0: Validation Loss = 0.6614402770996094
2026-03-12 17:33:58.160 | INFO     | modules.train:run_training:216 - Epoch 0: Validation Accuracy = 0.5020833333333333
2026-03-12 17:33:58.164 | INFO     | modules.train:run_training:217 - Epoch 0: Validation F1 = 0.34156378600823045
2026-03-12 17:33:58.197 | INFO     | modules.train:run_training:114 - Epoch 1 :
2026-03-12 17:34:01.352 | INFO     | modules.train:run_training:211 - Epoch 1: Train Loss = 0.6918763295659479
2026-03-12 17:34:01.352 | INFO     | modul

In [21]:
#Training with CLIP embeddings and using FLAVA multimodal embeddings
model=HeadClassifierFLAVAModel(with_clip_image=True,with_clip_text=True)
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
num_warmup_steps=int(0.1*n_steps)
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=1e-4)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_warmup_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
trainer=Train(model,loss_fn,optimizer,n_epochs,scheduler,device)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/multimodal_with_clip_savings",multimodal=True,with_clip=True)

2026-03-12 17:34:26.966 | INFO     | modules.train:run_training:114 - Epoch 0 :


2026-03-12 17:34:30.824 | INFO     | modules.train:run_training:211 - Epoch 0: Train Loss = 0.7605600908117475
2026-03-12 17:34:30.824 | INFO     | modules.train:run_training:212 - Epoch 0: Train Accuracy = 0.6127358490566037
2026-03-12 17:34:30.824 | INFO     | modules.train:run_training:213 - Epoch 0: Train F1 = 0.5446854926126455
2026-03-12 17:34:30.824 | INFO     | modules.train:run_training:215 - Epoch 0: Validation Loss = 0.8256111224492391
2026-03-12 17:34:30.838 | INFO     | modules.train:run_training:216 - Epoch 0: Validation Accuracy = 0.4979166666666667
2026-03-12 17:34:30.838 | INFO     | modules.train:run_training:217 - Epoch 0: Validation F1 = 0.3431415414142217
2026-03-12 17:34:30.840 | INFO     | modules.train:run_training:114 - Epoch 1 :
2026-03-12 17:34:34.645 | INFO     | modules.train:run_training:211 - Epoch 1: Train Loss = 0.6549331453611266
2026-03-12 17:34:34.645 | INFO     | modules.train:run_training:212 - Epoch 1: Train Accuracy = 0.61875
2026-03-12 17:34:34.